# NB1 – Thống kê (R1-C7, R1-C8, R2-C1, R2-C3)
**Runtime:** CPU · High-RAM  ·  **Thời gian ước tính:** 60–90 phút

Notebook tự chứa: code được nhúng sẵn (các ô `%%writefile`), không cần upload file.
Có thể chạy **song song** với các notebook khác. Chọn *Runtime → Run all*.
Kết quả được chép vào `MyDrive/PLOS_revision/NB1_CPU_stats_truncation/`.

Gom 24 file dự đoán → nối nguồn báo/năm → thống kê theo nguồn & năm → thống kê truncation theo từng tokenizer.

In [ ]:
# 1) Drive (kết quả được chép vào đây) + 2) đăng nhập Hugging Face (tài khoản BaoNhan)
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
RESULT_DIR = "/content/drive/MyDrive/PLOS_revision/NB1_CPU_stats_truncation"
os.makedirs(RESULT_DIR, exist_ok=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")      # Colab secret tên HF_TOKEN (nếu có)
except Exception:
    from huggingface_hub import login
    login()                                                 # nếu chưa có secret: dán token vào ô hiện ra
from huggingface_hub import whoami
try:
    print("Đã đăng nhập Hugging Face:", whoami()["name"])
except Exception:
    raise SystemExit("CHƯA ĐĂNG NHẬP Hugging Face. Bấm biểu tượng chìa khóa (Secrets) bên trái, "
                     "bật 'Notebook access' cho HF_TOKEN, rồi chạy lại ô này.")
!pip -q install datasets pyarrow scikit-learn

## Code (nhúng sẵn – không cần sửa)

In [ ]:
%%writefile common.py
"""Shared configuration for the PLOS ONE revision analyses (PONE-D-26-38656).

Every constant below is taken from the manuscript or from the Hugging Face
repositories listed by step0_inspect.py. Nothing here is estimated.
"""
from pathlib import Path

SEEDS = [42, 22, 202]
FIG_SEED_ORDER = [22, 42, 202]          # panel order of the original Fig 5
MAX_LEN = 256                           # model-specific token budget
N_CLASSES = 10

# Model display name -> (HF repo, short key used in folder/file names)
MODELS = {
    "WikiBERT":   ("BaoNhan/WikiBERT_Vietnamese_News_Classification", "wikibert"),
    "PhoBERT":    ("BaoNhan/PhoBERT_Vietnamese_News_Classification", "phobert"),
    "mBERT":      ("BaoNhan/mBERT_Vietnamese_News_Classification", "mbert"),
    "DistilBERT": ("BaoNhan/DistilBERT_Vietnamese_News_Classification", "distilbert"),
    "XLM-R":      ("BaoNhan/XLMR_Vietnamese_News_Classification", "xlmr"),
    "CafeBERT":   (None, "cafebert"),   # fine-tuned CafeBERT runs are not on the Hub
    "viBERT":     ("BaoNhan/viBERT_Vietnamese_News_Classification", "vibert"),
    "vELECTRA":   ("BaoNhan/vELECTRA_Vietnamese_News_Classification", "velectra"),
    "ViSoBERT":   ("BaoNhan/ViSoBERT_Vietnamese_News_Classification", "visobert"),
}
MODEL_ORDER = list(MODELS)  # same order as Table 2
# Public pre-trained checkpoints that were fine-tuned (verified to exist on the Hub;
# step4 also checks their sizes against the released fine-tuned configs).
BASE_CHECKPOINTS = {
    "WikiBERT":   "TurkuNLP/wikibert-base-vi-cased",
    "PhoBERT":    "vinai/phobert-base",
    "mBERT":      "google-bert/bert-base-multilingual-cased",
    "DistilBERT": "distilbert/distilbert-base-multilingual-cased",
    "XLM-R":      "FacebookAI/xlm-roberta-base",
    "CafeBERT":   "uitnlp/CafeBERT",
    "viBERT":     "FPTAI/vibert-base-cased",
    "vELECTRA":   "FPTAI/velectra-base-discriminator-cased",
    "ViSoBERT":   "uitnlp/visobert",
}
# Models whose seed-specific TCC was not retained: Fig 5 uses the Table 2 mean TCC
# for every seed of these models and the caption states this explicitly.
MEAN_TCC_FALLBACK = ["PhoBERT", "CafeBERT"]

# Category indices used by all prediction files (manuscript, Results section)
LABEL_NAMES = ["Education", "Entertainment", "Business", "Law", "Health",
               "World", "Sports", "Politics", "Automotive", "Lifestyle"]
# Test-set row totals per index, as reported in the manuscript
TEST_COUNTS = [6090, 9542, 12350, 6992, 9766, 13673, 12423, 13490, 4844, 5500]
N_TEST = 94670

# Table 2 of the submitted manuscript: mean (population SD) over seeds
TABLE2 = {  # loss, acc, P, R, F1, AUC, TCC
    "WikiBERT":   [0.20330, 0.93016, 0.92377, 0.92264, 0.92302, 0.99635, 3077.08],
    "PhoBERT":    [0.14761, 0.94779, 0.94124, 0.94375, 0.94241, 0.99794, 3360.89],
    "mBERT":      [0.16421, 0.94270, 0.93707, 0.93660, 0.93674, 0.99750, 3181.59],
    "DistilBERT": [0.17685, 0.93793, 0.93145, 0.93213, 0.93167, 0.99713, 1554.86],
    "XLM-R":      [0.15650, 0.94488, 0.93907, 0.94011, 0.93953, 0.99770, 3099.05],
    "CafeBERT":   [0.14729, 0.94916, 0.94391, 0.94445, 0.94413, 0.99800, 9335.51],
    "viBERT":     [0.20287, 0.92903, 0.92226, 0.92185, 0.92190, 0.99629, 3065.10],
    "vELECTRA":   [0.18201, 0.93791, 0.93174, 0.93189, 0.93161, 0.99710, 3192.59],
    "ViSoBERT":   [0.21663, 0.92617, 0.91819, 0.92075, 0.91934, 0.99604, 3201.82],
}
TABLE2_TCC_SD = {"WikiBERT": 3.46492, "PhoBERT": 4.53214, "mBERT": 1.24008,
                 "DistilBERT": 0.99322, "XLM-R": 1.41930, "CafeBERT": 148.46973,
                 "viBERT": 1.53178, "vELECTRA": 1.57443, "ViSoBERT": 0.44646}

COLORS = {"WikiBERT": "#4C78A8", "PhoBERT": "#F58518", "mBERT": "#54A24B",
          "DistilBERT": "#E45756", "XLM-R": "#B279A2", "CafeBERT": "#72B7B2",
          "viBERT": "#FF9DA6", "vELECTRA": "#9D755D", "ViSoBERT": "#BAB0AC"}

DATA = Path("revision_data")      # created by step1
OUT = Path("revision_results")    # created by step2/step3


def pred_path(model, seed):
    return DATA / "preds" / f"{model}_seed{seed}.npz"

In [ ]:
%%writefile step1_collect.py
"""STEP 1 - Download the released predictions/metrics and assemble them.

Runs on CPU. Requires `huggingface_hub`, `datasets`, `pandas`, `numpy`.
Output folder: revision_data/
  preds/<Model>_seed<s>.npz   true, pred, prob (aligned row by row)
  test_text.pkl           row, id, content of the 94,670 test articles
  metrics/<Model>.json        raw per-seed metric JSON(s) as released
  dataset_meta.pkl        label/url/source of BaoNhan/Vietnam_News_3_Sources
  step1_report.txt            every check performed and its outcome

The script STOPS with an explanation if any consistency check fails.
"""
import json
import sys

import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download

from common import (DATA, MODELS, N_TEST, SEEDS, TEST_COUNTS,
                    pred_path)

api = HfApi()
(DATA / "preds").mkdir(parents=True, exist_ok=True)
(DATA / "metrics").mkdir(parents=True, exist_ok=True)
report = []


def log(msg):
    print(msg)
    report.append(msg)


def fail(msg):
    log("STOP: " + msg)
    (DATA / "step1_report.txt").write_text("\n".join(report), encoding="utf-8")
    sys.exit(1)


# ---------------------------------------------------------------- locate files
locations = {}  # (model, seed) -> (repo, csv path)
metric_files = {}  # model -> list of (repo, path)
import os
ONLY = [m.strip() for m in os.environ.get("ONLY_MODELS", "").split(",") if m.strip()]
for model, (repo, key) in MODELS.items():
    if repo is None or (ONLY and model not in ONLY):
        continue
    files = api.list_repo_files(repo)
    for s in SEEDS:
        p = f"results_{key}/predictions_seed_{s}.csv"
        if p not in files:
            fail(f"{repo} has no {p}")
        locations[(model, s)] = (repo, p)
    metric_files[model] = [(repo, f) for f in files
                           if f.startswith(f"results_{key}/") and f.endswith(".json")]

metric_files["CafeBERT"] = []
log("CafeBERT: fine-tuned predictions are not released on the Hub - skipped")

# ---------------------------------------------------------------- predictions
PROB_COLS = [f"prob_class_{k}" for k in range(10)]
ref_true, ref_id, text_saved = None, None, False
for (model, s), (repo, path) in sorted(locations.items()):
    local = hf_hub_download(repo, path)
    header = pd.read_csv(local, nrows=0).columns.tolist()
    use = [c for c in header if c != "content"]
    df = pd.read_csv(local, usecols=use)
    if len(df) != N_TEST:
        fail(f"{model} seed {s}: {len(df)} rows, expected {N_TEST}")
    for c in ["true_label_0_9", "predicted_label_0_9"] + PROB_COLS:
        if c not in df.columns:
            fail(f"{model} seed {s}: column {c} missing; columns = {header}")
    true = df["true_label_0_9"].to_numpy(np.int8)
    pred = df["predicted_label_0_9"].to_numpy(np.int8)
    prob = df[PROB_COLS].to_numpy(np.float32)
    if ref_true is None:
        ref_true = true
        counts = np.bincount(true, minlength=10).tolist()
        if counts != TEST_COUNTS:
            fail(f"class counts {counts} differ from manuscript {TEST_COUNTS}")
        log(f"class counts match the manuscript: {counts}")
    elif not np.array_equal(true, ref_true):
        fail(f"{model} seed {s}: true-label order differs from the reference file")
    if "id" in df.columns:
        ids = df["id"].to_numpy()
        if ref_id is None:
            ref_id = ids
        elif not np.array_equal(ids, ref_id):
            fail(f"{model} seed {s}: id order differs from the reference file")
        if not text_saved:
            txt = pd.read_csv(local, usecols=["id", "content"])
            txt.insert(0, "row", np.arange(len(txt)))
            txt.to_pickle(DATA / "test_text.pkl")
            text_saved = True
            log(f"test text saved from {repo}/{path}")
    np.savez_compressed(pred_path(model, s), true=true, pred=pred, prob=prob)
    log(f"{model:10s} seed {s:3d}: OK  acc={np.mean(true == pred):.5f}  "
        f"id column={'id' in df.columns}")

missing = [(m, s) for m in MODELS for s in SEEDS if (m, s) not in locations and (not ONLY or m in ONLY)]
log(f"prediction files missing: {missing if missing else 'none'}")

# ---------------------------------------------------------------- metrics json
for model, lst in metric_files.items():
    merged = {}
    for repo, f in lst:
        merged[f"{repo}/{f}"] = json.load(open(hf_hub_download(repo, f), encoding="utf-8"))
    (DATA / "metrics" / f"{model}.json").write_text(
        json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")
    log(f"{model}: {len(lst)} metric JSON file(s) saved")

# ---------------------------------------------------------------- dataset meta
try:
    from datasets import load_dataset
    ds = load_dataset("BaoNhan/Vietnam_News_3_Sources")["train"].to_pandas()
    ds.insert(0, "row", np.arange(len(ds)))
    ds.to_pickle(DATA / "dataset_meta.pkl")
    log(f"dataset metadata saved: {len(ds)} rows, columns {list(ds.columns)}")
    if ref_id is not None:
        log(f"test id examples: {ref_id[:5].tolist()}  dtype={ref_id.dtype}")
except Exception as e:  # noqa: BLE001
    log(f"dataset metadata NOT saved: {e!r}")

(DATA / "step1_report.txt").write_text("\n".join(report), encoding="utf-8")
print("\nDone. Send revision_data/step1_report.txt.")

In [ ]:
%%writefile step1b_index.py
"""STEP 1b - Link the three private datasets row by row (CPU, high-RAM runtime).

  BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split  label(int), content  train/dev/test
  BaoNhan/Vietnamese_News_PhoBERT_Segmented        same rows, word-segmented content
  BaoNhan/Vietnam_News_3_Sources_Plus              url, label(str), content, source,
                                                   day, month, year (one split)

Output revision_data/corpus_index.pkl - one row per Cleaned article:
  split, pos, label (0-9 index used by the prediction files), source, year, url,
  plus_row (row in the Plus dataset, -1 if not found), h_exact, h_norm (content hashes)
and revision_data/step1b_report.txt. Every link is verified, nothing is assumed:
  * Cleaned and Segmented must have identical labels at every (split, pos)
  * the Cleaned test labels must equal the prediction-file labels (offset detected)
  * Plus rows are matched to Cleaned rows by an exact hash of the content
    (a whitespace-normalised hash is used only for rows without an exact match)
  * the category names implied by the match are checked against the manuscript
"""
import hashlib
import re
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download

from common import DATA, LABEL_NAMES, MODEL_ORDER, SEEDS, pred_path

CLEAN = "BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split"
SEG = "BaoNhan/Vietnamese_News_PhoBERT_Segmented"
PLUS = "BaoNhan/Vietnam_News_3_Sources_Plus"
SPLIT_SIZES = {"train": 757356, "dev": 94670, "test": 94670}
report = []
WS = re.compile(r"\s+")


def log(msg=""):
    print(msg)
    report.append(str(msg))


def stop(msg):
    log("STOP: " + msg)
    (DATA / "step1b_report.txt").write_text("\n".join(report), encoding="utf-8")
    sys.exit(1)


def h_exact(t):
    return hashlib.blake2b((t or "").encode("utf-8"), digest_size=8).hexdigest()


def h_norm(t):
    return h_exact(WS.sub(" ", t or "").strip())


def read_repo(repo, prefix, columns):
    """Stream a split's parquet shards; content is reduced to hashes immediately."""
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.startswith(f"data/{prefix}-") and f.endswith(".parquet"))
    out = []
    for f in files:
        pf = pq.ParquetFile(hf_hub_download(repo, f, repo_type="dataset"))
        for b in pf.iter_batches(batch_size=20000, columns=columns):
            d = b.to_pandas()
            if "content" in d:
                d["h_exact"] = d["content"].map(h_exact)
                d["h_norm"] = d["content"].map(h_norm)
                d = d.drop(columns="content")
            out.append(d)
    return pd.concat(out, ignore_index=True)


# -------------------------------------------------------- Cleaned and Segmented
parts = []
for split, n in SPLIT_SIZES.items():
    c = read_repo(CLEAN, split, ["label", "content"])
    s = read_repo(SEG, split, ["label"])
    if len(c) != n or len(s) != n:
        stop(f"{split}: Cleaned {len(c)} / Segmented {len(s)} rows, expected {n}")
    if not np.array_equal(c["label"].to_numpy(), s["label"].to_numpy()):
        stop(f"{split}: Cleaned and Segmented labels differ - rows are not aligned")
    c.insert(0, "split", split)
    c.insert(1, "pos", np.arange(n))
    parts.append(c)
    log(f"{split}: {n} rows, Cleaned/Segmented labels identical at every position; "
        f"label range {c['label'].min()}..{c['label'].max()}")
idx = pd.concat(parts, ignore_index=True)

# -------------------------------------------------------- label offset vs predictions
ref = next((pred_path(m, s) for m in MODEL_ORDER for s in SEEDS if pred_path(m, s).exists()), None)
if ref is None:
    stop("no prediction file found - run step1_collect.py first")
true = np.load(ref)["true"].astype(int)
test_lab = idx.loc[idx.split == "test", "label"].to_numpy()
offset = next((k for k in (0, 1) if np.array_equal(test_lab - k, true)), None)
if offset is None:
    stop("Cleaned test labels do not equal the prediction-file labels (offset 0 or 1)")
idx["label"] = idx["label"] - offset
log(f"Cleaned test split == prediction-file order; dataset label = index + {offset}")

# -------------------------------------------------------- Plus metadata
plus = read_repo(PLUS, "train", ["url", "label", "content", "source", "year"])
plus.insert(0, "plus_row", np.arange(len(plus)))
log(f"Plus: {len(plus)} rows; sources {plus.source.value_counts().to_dict()}")
for key in ("h_exact", "h_norm"):
    d = plus[key].duplicated(keep=False).sum()
    log(f"Plus rows sharing a {key} with another row: {d}")

m = idx.merge(plus.drop_duplicates("h_exact")[["h_exact", "plus_row"]], on="h_exact", how="left")
miss = m.plus_row.isna()
if miss.any():
    alt = plus.drop_duplicates("h_norm")[["h_norm", "plus_row"]].rename(columns={"plus_row": "pr2"})
    m = m.merge(alt, on="h_norm", how="left")
    m.loc[miss, "plus_row"] = m.loc[miss, "pr2"]
    m = m.drop(columns="pr2")
m["plus_row"] = m["plus_row"].fillna(-1).astype(int)
log(f"Cleaned rows matched to Plus: {(m.plus_row >= 0).mean():.4%} "
    f"(exact {(~miss).mean():.4%})")
meta = plus.set_index("plus_row")
ok = m.plus_row >= 0
for col in ("url", "source", "year"):
    m.loc[ok, col] = meta.loc[m.loc[ok, "plus_row"], col].to_numpy()
m.loc[ok, "plus_label"] = meta.loc[m.loc[ok, "plus_row"], "label"].to_numpy()

# -------------------------------------------------------- category-name check
ct = pd.crosstab(m.loc[ok, "label"], m.loc[ok, "plus_label"])
implied = ct.idxmax(axis=1).reindex(range(10)).tolist()
purity = (ct.max(axis=1) / ct.sum(axis=1)).round(4).tolist()
log(f"category name implied by Plus for index 0..9: {implied}")
log(f"share of rows agreeing with that name: {purity}")
log(f"manuscript names for index 0..9:          {LABEL_NAMES}")
log("NAMES AGREE" if implied == LABEL_NAMES else "NAMES DIFFER - report this before using them")
for split in SPLIT_SIZES:
    s = m[m.split == split]
    log(f"{split}: years {s.year.value_counts().sort_index().to_dict()}")

m.to_pickle(DATA / "corpus_index.pkl")
(DATA / "step1b_report.txt").write_text("\n".join(report), encoding="utf-8")
print("Done.")

In [ ]:
%%writefile step2_stats.py
"""STEP 2 - Statistics that need no GPU (Reviewer #1 C4, C5, C8; Reviewer #2 C1).

Reads revision_data/ (from step1) and writes revision_results/:
  step2_report.txt            all checks and headline numbers
  per_seed_metrics.csv        recomputed metrics for every model-seed run
  tcc_per_seed.csv            seed-specific TCC = train_duration + predict_duration
  Fig5.pdf / Fig5.tif         four-panel performance-cost figure (R1 C4)
  cafe_vs_phobert.csv         paired bootstrap + McNemar per seed and pooled (R1 C5)
  pairwise_bootstrap.csv      seed-averaged delta macro-F1 for all 36 pairs, Holm-adjusted
  per_class_f1.csv / .tex     per-category F1, mean (SD) over seeds (R1 C8)
  per_source_year.csv         macro-F1/accuracy per publisher and per year (R2 C1),
                              using revision_data/corpus_index.pkl from step1b

Optional input: manual_tcc.csv (columns model,seed,tcc_seconds) for runs whose
released JSON has no train_duration/predict_duration.
"""
import json
import sys
from itertools import combinations
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest, chi2
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

from common import (COLORS, DATA, FIG_SEED_ORDER, LABEL_NAMES, MODEL_ORDER,
                    N_CLASSES, OUT, SEEDS, TABLE2, TABLE2_TCC_SD, pred_path)

import os
B = int(os.environ.get("BOOT_B", 10_000))
RNG = np.random.default_rng(20261106)
OUT.mkdir(parents=True, exist_ok=True)
report = []


def log(msg=""):
    print(msg)
    report.append(str(msg))


def save_report():
    (OUT / "step2_report.txt").write_text("\n".join(report), encoding="utf-8")


# ------------------------------------------------------------------ load
runs = {}
for m in MODEL_ORDER:
    for s in SEEDS:
        p = pred_path(m, s)
        if p.exists():
            runs[(m, s)] = dict(np.load(p))
models = [m for m in MODEL_ORDER if all((m, s) in runs for s in SEEDS)]
log(f"models with all three seeds: {models}")
log(f"models skipped (missing runs): {[m for m in MODEL_ORDER if m not in models]}")
if not models:
    save_report()
    sys.exit("no complete model found - run step1 first")
TRUE = runs[(models[0], SEEDS[0])]["true"].astype(np.int64)
N = len(TRUE)


def macro_f1_from_conf(conf):
    conf = conf.reshape(N_CLASSES, N_CLASSES)  # rows = true, cols = pred
    tp = np.diag(conf)
    prec = np.divide(tp, conf.sum(0), out=np.zeros(N_CLASSES), where=conf.sum(0) > 0)
    rec = np.divide(tp, conf.sum(1), out=np.zeros(N_CLASSES), where=conf.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(N_CLASSES),
                   where=(prec + rec) > 0)
    return f1.mean()


# ------------------------------------------------------------------ 1. metrics
rows = []
for (m, s), r in runs.items():
    y, yhat = r["true"], r["pred"]
    p, rc, f, _ = precision_recall_fscore_support(y, yhat, average="macro", zero_division=0)
    _, _, fw, _ = precision_recall_fscore_support(y, yhat, average="weighted", zero_division=0)
    auc = roc_auc_score(y, r["prob"], multi_class="ovr", average="macro")
    rows.append(dict(model=m, seed=s, accuracy=np.mean(y == yhat), macro_precision=p,
                     macro_recall=rc, macro_f1=f, weighted_f1=fw, macro_auc=auc))
met = pd.DataFrame(rows)
met.to_csv(OUT / "per_seed_metrics.csv", index=False)
log("\n[1] Recomputed means vs Table 2 (difference in the 5th decimal is rounding):")
for m in models:
    g = met[met.model == m]
    t = TABLE2[m]
    log(f"  {m:10s} acc {g.accuracy.mean():.5f} (T2 {t[1]:.5f})  "
        f"F1 {g.macro_f1.mean():.5f} (T2 {t[4]:.5f})  AUC {g.macro_auc.mean():.5f} "
        f"(T2 {t[5]:.5f})  weighted-F1 {g.weighted_f1.mean():.5f}")

# ------------------------------------------------------------------ 2. TCC
def find_seed_block(obj, seed):
    """Return the dict holding seed-specific metrics, searching nested JSON."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k) in (f"seed_{seed}", str(seed), f"seed{seed}") and isinstance(v, dict):
                return v
        for v in obj.values():
            hit = find_seed_block(v, seed)
            if hit is not None:
                return hit
    return None


tcc = {}
for m in models:
    f = DATA / "metrics" / f"{m}.json"
    if not f.exists():
        continue
    merged = json.load(open(f, encoding="utf-8"))
    for s in SEEDS:
        blk = None
        for fname, content in merged.items():
            if m == "CafeBERT" and f"seed{s}" not in fname:
                continue  # one repository per seed
            blk = find_seed_block(content, s)
            if blk is None and m == "CafeBERT" and isinstance(content, dict):
                blk = content
            if blk and "train_duration" in blk and "predict_duration" in blk:
                tcc[(m, s)] = blk["train_duration"] + blk["predict_duration"]
                break
manual = Path("manual_tcc.csv")
if manual.exists():
    for _, r in pd.read_csv(manual).iterrows():
        tcc.setdefault((r["model"], int(r["seed"])), float(r["tcc_seconds"]))
        log(f"  manual TCC used for {r['model']} seed {int(r['seed'])}")
tdf = pd.DataFrame([dict(model=m, seed=s, tcc_seconds=v) for (m, s), v in tcc.items()])
tdf.to_csv(OUT / "tcc_per_seed.csv", index=False)
log("\n[2] Seed-specific TCC (s) vs Table 2 (mean and population SD):")
tcc_models = []
for m in models:
    v = [tcc.get((m, s)) for s in SEEDS]
    if None in v:
        log(f"  {m:10s} MISSING for seeds {[s for s, x in zip(SEEDS, v) if x is None]}")
        continue
    tcc_models.append(m)
    log(f"  {m:10s} " + "  ".join(f"s{s}={x:9.2f}" for s, x in zip(SEEDS, v)) +
        f"  mean={np.mean(v):9.2f} (T2 {TABLE2[m][6]:9.2f})  "
        f"sd={np.std(v):8.3f} (T2 {TABLE2_TCC_SD[m]:8.3f})")

# ------------------------------------------------------------------ 3. Fig 5
MARK = {"WikiBERT": "o", "PhoBERT": "s", "mBERT": "^", "DistilBERT": "D",
        "XLM-R": "P", "CafeBERT": "X", "viBERT": "v", "vELECTRA": "<", "ViSoBERT": ">"}
LEG = {"PhoBERT": "PhoBERT-base", "XLM-R": "XLM-R-base", "vELECTRA": "vELECTRA-base"}
missing_fig = [m for m in MODEL_ORDER if m not in tcc_models]
if missing_fig:
    log(f"\n[3] Fig 5 NOT drawn: no seed-specific TCC for {missing_fig}")
else:
    plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 9})
    fig, axes = plt.subplots(2, 2, figsize=(7.5, 6.6), sharex=True, sharey=True)
    panels = [(f"({c}) Seed {s}", s) for c, s in zip("abc", FIG_SEED_ORDER)]
    panels.append(("(d) Mean over the three seeds", None))
    for ax, (title, s) in zip(axes.ravel(), panels):
        ax.set_facecolor("#EAEAF2")
        ax.grid(color="white", linewidth=1)
        ax.set_axisbelow(True)
        xs, ys = [], []
        for m in MODEL_ORDER:
            f1s = [met[(met.model == m) & (met.seed == q)].macro_f1.item() * 100 for q in SEEDS]
            ts = [tcc[(m, q)] for q in SEEDS]
            if s is None:
                x, y = np.log2(np.mean(ts)), np.mean(f1s)
                lo = x - np.log2(np.mean(ts) - np.std(ts))
                hi = np.log2(np.mean(ts) + np.std(ts)) - x
                ax.errorbar(x, y, xerr=[[lo], [hi]], yerr=np.std(f1s), fmt="none",
                            ecolor="black", elinewidth=0.8, capsize=2, zorder=2)
            else:
                x, y = np.log2(tcc[(m, s)]), f1s[SEEDS.index(s)]
            xs.append(x)
            ys.append(y)
            ax.scatter(x, y, marker=MARK[m], s=46, color=COLORS[m], edgecolor="black",
                       linewidth=0.6, zorder=3, label=LEG.get(m, m))
        ax.axhline(np.median(ys), ls="--", lw=0.8, color="grey")
        ax.axvline(np.median(xs), ls="--", lw=0.8, color="grey")
        ax.set_title(title, fontweight="bold", fontsize=9)
    for ax in axes[1]:
        ax.set_xlabel(r"$\log_2$ TCC (s)", fontweight="bold")
    for ax in axes[:, 0]:
        ax.set_ylabel("Macro-F1 (%)", fontweight="bold")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=5, fontsize=8, frameon=True)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    fig.savefig(OUT / "Fig5.pdf", bbox_inches="tight")
    fig.savefig(OUT / "Fig5.tif", dpi=400, bbox_inches="tight",
                pil_kwargs={"compression": "tiff_lzw"})
    log("\n[3] Fig 5 saved (panels a-c: seed-specific TCC and macro-F1; "
        "panel d: mean TCC vs mean macro-F1, error bars = population SD; "
        "dashed lines = medians across the nine checkpoints)")

# ------------------------------------------------------------------ 4. tests
log("\n[4] Paired tests")
codes = {k: TRUE * N_CLASSES + v["pred"].astype(np.int64) for k, v in runs.items()
         if k[0] in models}
obs_f1 = {k: macro_f1_from_conf(np.bincount(c, minlength=100)) for k, c in codes.items()}
pairs = list(combinations(models, 2))
boot = {k: np.empty(B) for k in codes}
for b in range(B):
    w = np.bincount(RNG.integers(0, N, N), minlength=N)
    for k, c in codes.items():
        boot[k][b] = macro_f1_from_conf(np.bincount(c, weights=w, minlength=100))


def summarize(delta_obs, delta_boot):
    lo, hi = np.percentile(delta_boot, [2.5, 97.5])
    p = min(1.0, 2 * min(np.mean(delta_boot <= 0), np.mean(delta_boot >= 0)))
    return delta_obs, lo, hi, p


rows = []
if "CafeBERT" in models and "PhoBERT" in models:
    for s in SEEDS:
        a, bm = runs[("CafeBERT", s)], runs[("PhoBERT", s)]
        ca, cb = a["pred"] == a["true"], bm["pred"] == bm["true"]
        n01, n10 = int(np.sum(ca & ~cb)), int(np.sum(~ca & cb))
        d, lo, hi, p = summarize(obs_f1[("CafeBERT", s)] - obs_f1[("PhoBERT", s)],
                                 boot[("CafeBERT", s)] - boot[("PhoBERT", s)])
        chi = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
        rows.append(dict(seed=s, delta_macro_f1_pp=d * 100, ci_low_pp=lo * 100,
                         ci_high_pp=hi * 100, bootstrap_p=p,
                         delta_accuracy_pp=(ca.mean() - cb.mean()) * 100,
                         cafe_only_correct=n01, phobert_only_correct=n10,
                         net_articles=n01 - n10,
                         mcnemar_exact_p=binomtest(n01, n01 + n10, 0.5).pvalue,
                         mcnemar_chi2_cc=chi, mcnemar_chi2_p=chi2.sf(chi, 1)))
    d_obs = np.mean([obs_f1[("CafeBERT", s)] - obs_f1[("PhoBERT", s)] for s in SEEDS])
    d_boot = np.mean([boot[("CafeBERT", s)] - boot[("PhoBERT", s)] for s in SEEDS], axis=0)
    d, lo, hi, p = summarize(d_obs, d_boot)
    rows.append(dict(seed="mean", delta_macro_f1_pp=d * 100, ci_low_pp=lo * 100,
                     ci_high_pp=hi * 100, bootstrap_p=p))
    cvp = pd.DataFrame(rows)
    cvp.to_csv(OUT / "cafe_vs_phobert.csv", index=False)
    log("  CafeBERT - PhoBERT (percentage points), B = %d:" % B)
    log(cvp.to_string(index=False))
else:
    log("  CafeBERT vs PhoBERT skipped: one of them has no complete prediction set")

pw = []
for a, bm in pairs:
    d_obs = np.mean([obs_f1[(a, s)] - obs_f1[(bm, s)] for s in SEEDS])
    d_boot = np.mean([boot[(a, s)] - boot[(bm, s)] for s in SEEDS], axis=0)
    d, lo, hi, p = summarize(d_obs, d_boot)
    pw.append(dict(model_a=a, model_b=bm, delta_macro_f1_pp=d * 100,
                   ci_low_pp=lo * 100, ci_high_pp=hi * 100, p=p))
pw = pd.DataFrame(pw).sort_values("p").reset_index(drop=True)
k = len(pw)
pw["p_holm"] = np.minimum(1, np.maximum.accumulate(pw.p * (k - np.arange(k))))
pw.to_csv(OUT / "pairwise_bootstrap.csv", index=False)
log(f"  all {k} pairs written to pairwise_bootstrap.csv; "
    f"{int((pw.p_holm < 0.05).sum())} remain significant after Holm (alpha 0.05)")

# ------------------------------------------------------------------ 5. per class
pc = []
for (m, s), r in runs.items():
    if m not in models:
        continue
    _, _, f, _ = precision_recall_fscore_support(r["true"], r["pred"], labels=range(10),
                                                 average=None, zero_division=0)
    for c in range(N_CLASSES):
        pc.append(dict(model=m, seed=s, category=LABEL_NAMES[c], f1=f[c]))
pc = pd.DataFrame(pc)
pc.to_csv(OUT / "per_class_f1.csv", index=False)
tab = pc.groupby(["category", "model"]).f1.agg(["mean", lambda x: np.std(x)]).reset_index()
tab.columns = ["category", "model", "mean", "sd"]
cells = tab.assign(cell=lambda d: d["mean"].map(lambda v: f"{v*100:.2f}") + " (" +
                   d["sd"].map(lambda v: f"{v*100:.2f}") + ")")
wide = cells.pivot(index="category", columns="model", values="cell")[models]
wide = wide.loc[[c for c in LABEL_NAMES]]
(OUT / "per_class_f1.tex").write_text(wide.to_latex(escape=True), encoding="utf-8")
log("\n[5] Per-category F1 (%), mean (SD) over seeds:")
log(wide.to_string())

# ------------------------------------------------------------------ 6. sources
log("\n[6] Breakdown by publisher and by publication year (original test split)")
try:
    ci = pd.read_pickle(DATA / "corpus_index.pkl")
    t = ci[ci.split == "test"].sort_values("pos")
    if not np.array_equal(t["label"].to_numpy(), TRUE):
        raise ValueError("corpus_index test labels are not in prediction order - rerun step1b")
    log(f"  test articles with publisher/year metadata: {(t.plus_row >= 0).mean():.4%}")
    out = []
    for col in ("source", "year"):
        vals = t[col].to_numpy()
        for v in sorted(pd.Series(vals).dropna().unique()):
            mask = vals == v
            for m in models:
                f1s = [precision_recall_fscore_support(runs[(m, s)]["true"][mask], runs[(m, s)]["pred"][mask],
                                                       average="macro", zero_division=0)[2] for s in SEEDS]
                accs = [np.mean(runs[(m, s)]["true"][mask] == runs[(m, s)]["pred"][mask]) for s in SEEDS]
                out.append(dict(by=col, group=v, model=m, n=int(mask.sum()), macro_f1_mean=np.mean(f1s),
                                macro_f1_sd=np.std(f1s), acc_mean=np.mean(accs), acc_sd=np.std(accs)))
    out = pd.DataFrame(out)
    out.to_csv(OUT / "per_source_year.csv", index=False)
    for col in ("source", "year"):
        log(out[out.by == col].pivot(index="model", columns="group", values="macro_f1_mean")
            .loc[models].mul(100).round(2).to_string())
except Exception as e:  # noqa: BLE001
    log(f"  skipped: {e}")

save_report()
print("\nDone. Send the whole revision_results/ folder.")

In [ ]:
%%writefile step3_truncation.py
"""STEP 3 - Model-specific truncation audit on the test split (R1 C7, R2 C3).

Runs on CPU (a GPU is not needed). Uses the tokenizer saved with each released
checkpoint (seed 42).

Outputs in revision_results/:
  truncation_stats.csv / .tex     per checkpoint: token-length distribution,
                                  number and % of truncated articles, retained share
  truncation_performance.csv      macro-F1 / accuracy on truncated vs. untruncated
                                  articles (mean, population SD over seeds)
  token_lengths.npz               per-article token counts (reused by later steps)
  step3_report.txt

Inputs are the test splits of BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split
(eight models) and BaoNhan/Vietnamese_News_PhoBERT_Segmented (PhoBERT).
"""
import os
import sys

import numpy as np
import pandas as pd
from huggingface_hub import HfApi
from sklearn.metrics import precision_recall_fscore_support
from transformers import AutoTokenizer

from common import (BASE_CHECKPOINTS, DATA, MAX_LEN, MODEL_ORDER, MODELS, OUT, SEEDS,
                    pred_path)

OUT.mkdir(parents=True, exist_ok=True)
report = []


def log(msg=""):
    print(msg)
    report.append(str(msg))


def tokenizer_location(model):
    repo, key = MODELS[model]
    if repo is not None:
        return repo, f"{key}_checkpoints_seed_42/checkpoint-8877"
    return BASE_CHECKPOINTS[model], None  # CafeBERT: tokenizer of the public checkpoint


# ------------------------------------------------------------ model inputs
# Same test texts that were used in training: the Cleaned split for eight models,
# the word-segmented split for PhoBERT (datasets named by the authors).
CLEAN = "BaoNhan/Vietnamese_News_Cleaned_3_Sources_Split"
SEG = "BaoNhan/Vietnamese_News_PhoBERT_Segmented"


def load_test(repo):
    import pyarrow.parquet as pq
    from huggingface_hub import hf_hub_download
    files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset")
                   if f.startswith("data/test-") and f.endswith(".parquet"))
    tab = pd.concat([pq.read_table(hf_hub_download(repo, f, repo_type="dataset")).to_pandas()
                     for f in files], ignore_index=True)
    true = next(np.load(pred_path(m, s))["true"] for m in MODEL_ORDER for s in SEEDS
                if pred_path(m, s).exists()).astype(int)
    lab = tab["label"].to_numpy()
    if not any(np.array_equal(lab - k, true) for k in (0, 1)):
        sys.exit(f"{repo}: test labels are not in prediction-file order")
    return tab["content"].fillna("").astype(str).tolist()


texts_clean = load_test(CLEAN)
texts_seg = load_test(SEG)
log(f"test articles: {len(texts_clean)} (Cleaned) / {len(texts_seg)} (PhoBERT-segmented)")

cache = OUT / "token_lengths.npz"
lengths = dict(np.load(cache)) if cache.exists() else {}
n_special = {}
for m in MODEL_ORDER:
    repo, sub = tokenizer_location(m)
    tok = (AutoTokenizer.from_pretrained(repo, subfolder=sub) if sub
           else AutoTokenizer.from_pretrained(repo))
    n_special[m] = tok.num_special_tokens_to_add(pair=False)
    if m in lengths:
        continue
    inp = texts_seg if m == "PhoBERT" else texts_clean
    lens = []
    for i in range(0, len(inp), 2000):
        enc = tok(inp[i:i + 2000], add_special_tokens=True, truncation=False,
                  return_attention_mask=False, verbose=False)["input_ids"]
        lens += [len(x) for x in enc]
    lengths[m] = np.array(lens)
    np.savez_compressed(cache, **lengths)
    log(f"{m}: tokenizer {repo}/{sub} ({type(tok).__name__}), "
        f"{n_special[m]} special tokens")

# ------------------------------------------------------------ statistics
rows = []
for m in MODEL_ORDER:
    L = lengths[m]
    trunc = L > MAX_LEN
    kept = np.minimum(1.0, (MAX_LEN - n_special[m]) / np.maximum(L - n_special[m], 1))
    q1, q3 = np.percentile(kept[trunc], [25, 75]) if trunc.any() else (1.0, 1.0)
    rows.append(dict(
        model=m, mean_tokens=L.mean(), median_tokens=np.median(L),
        p90_tokens=np.percentile(L, 90), n_truncated=int(trunc.sum()),
        pct_truncated=100 * trunc.mean(),
        mean_retained_pct_all=100 * kept.mean(),
        median_retained_pct_truncated=100 * np.median(kept[trunc]) if trunc.any() else 100,
        iqr_retained_pct_truncated=f"{100*q1:.1f}-{100*q3:.1f}"))
st = pd.DataFrame(rows)
st.to_csv(OUT / "truncation_stats.csv", index=False)
(OUT / "truncation_stats.tex").write_text(
    st.to_latex(index=False, float_format="%.1f"), encoding="utf-8")
log("\nTruncation statistics (test split, budget = %d tokens incl. special tokens):" % MAX_LEN)
log(st.round(1).to_string(index=False))

# ------------------------------------------------------------ performance
perf = []
masks = {m: lengths[m] > MAX_LEN for m in MODEL_ORDER}
all_trunc = np.logical_and.reduce(list(masks.values()))
none_trunc = ~np.logical_or.reduce(list(masks.values()))
log(f"\narticles truncated for all nine checkpoints: {all_trunc.sum()}; "
    f"for none: {none_trunc.sum()}")
for m in MODEL_ORDER:
    if not all(pred_path(m, s).exists() for s in SEEDS):
        log(f"{m}: predictions missing, performance split skipped")
        continue
    for name, mask in [("own_truncated", masks[m]), ("own_untruncated", ~masks[m]),
                       ("truncated_for_all", all_trunc), ("truncated_for_none", none_trunc)]:
        f1s, accs = [], []
        for s in SEEDS:
            r = np.load(pred_path(m, s))
            y, yhat = r["true"][mask], r["pred"][mask]
            f1s.append(precision_recall_fscore_support(y, yhat, average="macro",
                                                       zero_division=0)[2])
            accs.append(np.mean(y == yhat))
        perf.append(dict(model=m, subset=name, n=int(mask.sum()),
                         macro_f1_mean=np.mean(f1s), macro_f1_sd=np.std(f1s),
                         acc_mean=np.mean(accs), acc_sd=np.std(accs)))
perf = pd.DataFrame(perf)
perf.to_csv(OUT / "truncation_performance.csv", index=False)
log(perf.pivot(index="model", columns="subset", values="macro_f1_mean")
    .mul(100).round(2).to_string())
(OUT / "step3_report.txt").write_text("\n".join(report), encoding="utf-8")
print("\nDone.")

## Dùng lại kết quả cũ trên Drive

In [ ]:
# Dùng lại kết quả của lần chạy trước trên Drive (nếu có) để bỏ qua các bước đã xong
import shutil, glob, os
for d in ["revision_data", "revision_data/preds", "revision_results"]:
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f"{RESULT_DIR}/{d}/*"):
        if os.path.isfile(f) and not os.path.exists(f"{d}/{os.path.basename(f)}"):
            shutil.copy(f, f"{d}/")
have = lambda p: os.path.exists(p)
print("preds:", len(glob.glob("revision_data/preds/*.npz")), "| corpus_index:", have("revision_data/corpus_index.pkl"),
      "| step2:", have("revision_results/step2_report.txt"))

## Chạy

In [ ]:
if len(glob.glob("revision_data/preds/*.npz")) < 24:
    !python step1_collect.py
if not have("revision_data/corpus_index.pkl"):
    !python step1b_index.py
if not have("revision_results/step2_report.txt"):
    !python step2_stats.py
!python step3_truncation.py

## Lưu kết quả

In [ ]:
# Chép kết quả sang Drive và tạo file zip để tải về
import shutil, glob
for d in ["revision_results", "revision_data"]:
    for f in glob.glob(f"{d}/*.txt") + glob.glob(f"{d}/*.csv") + glob.glob(f"{d}/*.tex") + \
             glob.glob(f"{d}/*.pdf") + glob.glob(f"{d}/*.tif") + glob.glob(f"{d}/*.pkl"):
        if os.path.getsize(f) < 500e6:
            os.makedirs(f"{RESULT_DIR}/{d}", exist_ok=True)
            shutil.copy(f, f"{RESULT_DIR}/{d}/")
os.makedirs(f"{RESULT_DIR}/revision_data/preds", exist_ok=True)
for f in glob.glob("revision_data/preds/*.npz"):
    shutil.copy(f, f"{RESULT_DIR}/revision_data/preds/")
print("Đã chép vào", RESULT_DIR)